<a href="https://colab.research.google.com/github/pavangavara12/24P31A1212_DriveReady_OPC_2026/blob/main/Day_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:

!pip install -q langchain langchain-core langchain-groq langgraph pydantic


In [51]:

catalogue = {
    "plum cake": {"price": 500},
    "chocolate cake": {"price": 650},
    "vanilla cake": {"price": 550},
    "red velvet cake": {"price": 750},
    "cupcake": {"price": 80}
}

print(catalogue)


{'plum cake': {'price': 500}, 'chocolate cake': {'price': 650}, 'vanilla cake': {'price': 550}, 'red velvet cake': {'price': 750}, 'cupcake': {'price': 80}}


In [52]:

from langchain_core.tools import tool

@tool
def get_product(item: str) -> dict:
    """Find product details and price from the Sweet Crumbs bakery catalogue."""

    return catalogue.get(item.lower().strip(), {})


@tool
def get_price(item: str, quantity: int = 1) -> int:
    """Calculate the total price of a bakery product for the requested quantity."""

    product = catalogue.get(item.lower().strip(), {})

    if not product:
        return 0

    return product["price"] * quantity


In [53]:

print(get_product.invoke({"item": "plum cake"}))

print(get_price.invoke({
    "item": "plum cake",
    "quantity": 2
}))


{'price': 500}
1000


In [54]:

from typing import Literal
from pydantic import BaseModel, Field

class BakeryOrder(BaseModel):
    """A bakery order extracted from a customer's message."""

    name: str = Field(
        description="Customer's name"
    )

    item: Literal[
        "plum cake",
        "chocolate cake",
        "vanilla cake",
        "red velvet cake",
        "cupcake"
    ] = Field(description="Bakery product")

    qty: int = Field(
        description="Number of items ordered",
        ge=1
    )

    date: str = Field(
        description="Customer's requested pickup date"
    )

    eggless: bool = Field(
        description="Whether the customer wants eggless products"
    )

print("BakeryOrder class created successfully!")


BakeryOrder class created successfully!


In [55]:

import os
from google.colab import userdata
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


In [56]:

extractor = llm.with_structured_output(BakeryOrder)

message = """
Hi, this is Ravi.
I need two chocolate cakes for tomorrow.
Please make both eggless.
"""

order = extractor.invoke(message)

print(order)


name='Ravi' item='chocolate cake' qty=2 date='2026-11-28' eggless=True


In [57]:

messages = [
    """hey im ravi need 2 choc cakes tmrw eggless pls""",
    """Hello, this is Priya. I want 3 cupcakes for 28 November. Regular is fine.""",
    """my name is arun, one plum cake for sunday no egg please""",
    """hi this is sneha. need 4 vanilla cakes on 30 nov. make it eggless""",
    """Rahul here!! 5 red velvet cakes for tomorrow, normal cakes not eggless."""
]

for i, msg in enumerate(messages, 1):
    print(f"\nCUSTOMER MESSAGE {i}")
    order = extractor.invoke(msg)
    print(order.model_dump())



CUSTOMER MESSAGE 1
{'name': 'Ravi', 'item': 'chocolate cake', 'qty': 2, 'date': '2026-11-28', 'eggless': True}

CUSTOMER MESSAGE 2
{'name': 'Priya', 'item': 'cupcake', 'qty': 3, 'date': '28 November', 'eggless': False}

CUSTOMER MESSAGE 3
{'name': 'arun', 'item': 'plum cake', 'qty': 1, 'date': 'Sunday', 'eggless': True}

CUSTOMER MESSAGE 4
{'name': 'Sneha', 'item': 'vanilla cake', 'qty': 4, 'date': '30 Nov', 'eggless': True}

CUSTOMER MESSAGE 5
{'name': 'Rahul', 'item': 'red velvet cake', 'qty': 5, 'date': '2026-11-28', 'eggless': False}


In [58]:

try:
    bad_order = BakeryOrder(
        name="Ravi",
        item="donut",
        qty=0,
        date="tomorrow",
        eggless=True
    )

except Exception as e:
    print("VALIDATION ERROR:")
    print(e)


VALIDATION ERROR:
2 validation errors for BakeryOrder
item
  Input should be 'plum cake', 'chocolate cake', 'vanilla cake', 'red velvet cake' or 'cupcake' [type=literal_error, input_value='donut', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error
qty
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/greater_than_equal


In [59]:

!pip install -q gspread


In [60]:

from google.colab import auth

auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()

gc = gspread.authorize(creds)

print("Google Sheets connected successfully!")


Google Sheets connected successfully!


In [61]:

sh = gc.create("Sweet Crumbs Orders")

ws = sh.sheet1

print("Spreadsheet created!")
print("Sheet URL:", sh.url)


Spreadsheet created!
Sheet URL: https://docs.google.com/spreadsheets/d/1WhnDpvG1ulWyABat2Mqj7ZOim8dbK623_u-1F8Eb4Hk


In [62]:

ws.append_row([
    "name",
    "item",
    "qty",
    "date",
    "eggless",
    "total"
])

print("Column headers added!")


Column headers added!


In [63]:

from langchain_core.tools import tool

@tool
def log_order(
    name: str,
    item: str,
    qty: int,
    date: str,
    eggless: bool
) -> str:
    """
    Save a confirmed bakery order to Google Sheets.
    Use only when customer name, product, quantity
    and pickup date are available.
    """

    try:
        # Validate order using Pydantic
        order = BakeryOrder(
            name=name,
            item=item,
            qty=qty,
            date=date,
            eggless=eggless
        )

        # Calculate total price
        total = catalogue[order.item]["price"] * order.qty

        # Save order to Google Sheets
        ws.append_row([
            order.name,
            order.item,
            order.qty,
            order.date,
            "yes" if order.eggless else "no",
            total
        ])

        return f"Order saved! Total: Rs. {total}"

    except Exception as e:
        return f"Could not save order: {e}"


In [64]:

result = log_order.invoke({
    "name": "Arun",
    "item": "plum cake",
    "qty": 2,
    "date": "Saturday",
    "eggless": True
})

print(result)


Order saved! Total: Rs. 1000


In [65]:

from langchain.agents import create_agent

SYSTEM = """
You are the friendly assistant for Sweet Crumbs Home Bakery.

Always use get_price for product prices.
Use log_order to save confirmed customer orders.
Never guess prices.

Before saving an order, make sure you have:
customer name, product, quantity and pickup date.
Ask the customer if any information is missing.
"""

agent = create_agent(
    model=llm,
    tools=[get_product, get_price, log_order],
    system_prompt=SYSTEM
)

print("Bakery agent created successfully!")


Bakery agent created successfully!


In [66]:

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of 2 plum cakes?"
        }
    ]
})

print(result["messages"][-1].content)


The total price for 2 plum cakes is **₹1,000**.


In [67]:

@tool
def tool1(item: str, quantity: int = 1) -> int:
    """does stuff"""

    product = catalogue.get(item.lower().strip(), {})

    if not product:
        return 0

    return product["price"] * quantity

print("Bad tool created!")


Bad tool created!


In [68]:

@tool
def get_price(item: str, quantity: int = 1) -> int:
    """
    Calculate the total price of a bakery product
    for the requested quantity.
    """

    product = catalogue.get(item.lower().strip(), {})

    if not product:
        return 0

    return product["price"] * quantity

print("Original tool restored!")


Original tool restored!


In [69]:

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of 3 chocolate cakes?"
        }
    ]
})


In [70]:

for m in result["messages"]:
    m.pretty_print()


================================ Human Message =================================

What is the price of 3 chocolate cakes?
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_b2e77a0d-1a03-4f48-98b3-cf5d32dadf5e)
 Call ID: fc_b2e77a0d-1a03-4f48-98b3-cf5d32dadf5e
  Args:
    item: chocolate cake
    quantity: 3
================================= Tool Message =================================
Name: get_price

1950
================================== Ai Message ==================================

The total price for 3 chocolate cakes is **1950**.


In [71]:

from langchain_core.messages import AIMessage, ToolMessage

tool_calls = 0
model_calls = 0

for message in result["messages"]:

    if isinstance(message, AIMessage):
        model_calls += 1
        tool_calls += len(message.tool_calls)

    elif isinstance(message, ToolMessage):
        pass

print("Total LLM calls:", model_calls)
print("Total tool calls:", tool_calls)


Total LLM calls: 2
Total tool calls: 1
